# Main study: Base, RAG, CAG and QLoRA on TAT-QA

One notebook for the whole experiment. Run the cells **from top to bottom**.

**Before you start**
- **Google Colab:** Runtime → Change runtime type → **T4 GPU** → Save.
- **Kaggle** (free 30 GPU hours a week): Settings panel on the right → Accelerator → **GPU T4 x2**, and Internet → **On**. (Your Kaggle account needs a verified phone number for both.) Do not choose P100.

**Good to know**
- Every model and strategy saves to its **own file** (`results/raw/<model>__<strategy>.csv`). Nothing is overwritten.
- Every run cell **continues where it stopped**. Running a finished cell again just reports *0 to run*.
- **Progress is saved to GitHub automatically** about every 5 minutes while a model runs, and when it finishes. If the session disconnects (the machine is wiped), at most the last few minutes are lost.
- **After a disconnect:** run the GPU check, *Setup A, B, C* and *Step 1* again, then the cell you were on. It downloads the saved results and continues from the next unanswered question.
- Do **not** use File → Save a copy in GitHub. Use *Step 7* to save results.

In [ ]:
# Works on Colab and Kaggle. You should see 'Tesla T4'.
# If it says NO GPU: Colab -> Runtime -> Change runtime type -> T4 GPU;
#                    Kaggle -> Settings -> Accelerator -> GPU T4 x2.
import os
ON_KAGGLE = os.path.exists('/kaggle')
PROJECT = '/kaggle/working/project' if ON_KAGGLE else '/content/project'
# Kaggle gives two T4s; use only one so speed and memory match a single T4 everywhere.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!nvidia-smi -L
import torch
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab')
print('GPU ready:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change the accelerator first')

## Setup A. Get the code and the results saved so far
Your GitHub token (starts with `ghp_`) is kept **only for this session**, so results can be saved to GitHub automatically. It is never written to a file.

**Kaggle tip:** instead of pasting it every time, add it once under *Add-ons → Secrets* with the name `GITHUB_TOKEN` (and your Hugging Face token as `HF_TOKEN`), and tick both for this notebook.

In [ ]:
import getpass, os
REPO = "github.com/faheemjabbar39-hub/llm-knowledge-integration.git"
def secret(name):
    """Kaggle Secret if one exists, otherwise ask."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return getpass.getpass(f"{name}: ")
os.environ['GITHUB_TOKEN'] = gh_token = secret('GITHUB_TOKEN').strip()
if not os.path.exists(PROJECT):
    !git clone -q https://{gh_token}@{REPO} {PROJECT}
    !git -C {PROJECT} remote set-url origin https://{REPO}
else:
    !git -C {PROJECT} pull -q --rebase --autostash -X theirs https://{gh_token}@{REPO} main
%cd {PROJECT}
!git log --oneline -1
print('Results already saved:')
!ls results/raw

## Setup B. Install packages and test the code (about 4 minutes)
The last line should say **31 passed**.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!python -m pytest -q 2>&1 | tail -2
import sys, os
for p in [os.path.join(os.getcwd(), 'src'), '/kaggle/working/project/src', '/content/project/src']:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)   # lets this notebook import the project code (kis) without a restart


## Setup C. Log in to Hugging Face
Paste your Hugging Face **Read** token (it starts with `hf_`). Llama needs the access you already have; Mistral and Qwen are open to everyone.

In [ ]:
from huggingface_hub import login
login(token=secret('HF_TOKEN').strip())

## Step 1. Prepare the data (1 minute)
Downloads TAT-QA and builds the files. The official splits are kept:
- **train** → used only to train QLoRA
- **dev** → used only for the checks in Step 2
- **test** → 504 questions used for all final results

The test report extracts are put into 3 groups ("blocks") of 28. Each block is one small knowledge base: RAG searches it, CAG preloads it.

In [ ]:
!python scripts/01_prepare_tatqa.py

## Step 2. Check the setup (about 3 minutes)
**A.** How often does search find the right report extract? (needs to be above 60%, ideally above 80%)

**B.** Does every block fit in each model's memory limit?

Both must say **GOOD / OK** before you continue. If not, stop and ask.

In [ ]:
!python scripts/02_check_setup.py

## Step 3. Smoke test (about 10 minutes)
30 questions through Base, RAG and CAG on Llama-3.1-8B, to confirm everything runs. These 30 answers are kept, so the full run in Step 4 does not repeat them.

In [ ]:
!python scripts/03_run_eval.py --model llama8b --strategy base rag cag --limit 30

In [ ]:
# Look at a few answers from each strategy.
import pandas as pd
pd.set_option('display.max_colwidth', 70)
import os
for s in ['base', 'rag', 'cag']:
    path = f'results/raw/llama8b__{s}.csv'
    if not os.path.exists(path):
        print(f'{s.upper()}: no results yet - the cell above did not finish. Read its last lines for the reason.')
        continue
    d = pd.read_csv(path)
    print(f"\n{s.upper()}: {d.correct.mean():.0%} correct, {d.hallucinated.mean():.0%} wrong answers, "
          f"{d.abstained.mean():.0%} said I don't know, median {d.latency_ms.median()/1000:.1f}s")
    display(d[['question', 'gold', 'final_answer', 'correct']].head(6))

## Step 4. Base, RAG and CAG: one cell per model and strategy
Each cell runs one strategy on all 504 test questions and saves to its own file. Base takes roughly 15–20 minutes per model; RAG and CAG take longer because the model reads documents.

### Llama-3.1-8B

In [ ]:
!python scripts/03_run_eval.py --model llama8b --strategy base

In [ ]:
!python scripts/03_run_eval.py --model llama8b --strategy rag

In [ ]:
!python scripts/03_run_eval.py --model llama8b --strategy cag

### Mistral-7B

In [ ]:
!python scripts/03_run_eval.py --model mistral7b --strategy base

In [ ]:
!python scripts/03_run_eval.py --model mistral7b --strategy rag

In [ ]:
!python scripts/03_run_eval.py --model mistral7b --strategy cag

### Qwen2.5-7B

In [ ]:
!python scripts/03_run_eval.py --model qwen7b --strategy base

In [ ]:
!python scripts/03_run_eval.py --model qwen7b --strategy rag

In [ ]:
!python scripts/03_run_eval.py --model qwen7b --strategy cag

### Step 4 results: one cell per strategy
Each cell shows how good that strategy is for every model that has finished, compared with Base (and CAG also with RAG): accuracy, wrong answers, "I don't know", speed, memory, cost, results by kind of question, significance tests and a chart. They work even if some runs are not finished.

In [ ]:
# Base results
import sys, os
for p in [os.path.join(os.getcwd(), 'src'), '/kaggle/working/project/src', '/content/project/src']:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)   # lets this notebook import the project code (kis) without a restart
from kis.report import show
show("base")

In [ ]:
# RAG results
import sys, os
for p in [os.path.join(os.getcwd(), 'src'), '/kaggle/working/project/src', '/content/project/src']:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)   # lets this notebook import the project code (kis) without a restart
from kis.report import show
show("rag")

In [ ]:
# CAG results
import sys, os
for p in [os.path.join(os.getcwd(), 'src'), '/kaggle/working/project/src', '/content/project/src']:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)   # lets this notebook import the project code (kis) without a restart
from kis.report import show
show("cag")

## Step 5. QLoRA: one cell per model
Each cell does three things:
1. **Trains** small adapter layers on 1,202 TAT-QA *training* questions (roughly 1 to 1.5 hours on a T4). If the adapter already exists, this is skipped.
2. **Tests** the fine-tuned model on the 504 test questions, with the same documents RAG finds.
3. **Extra check:** tests it again with **no documents**, to show what fine-tuning alone can do.

Run **Step 7** after each model: it also saves the trained adapter.

### 5a. Llama-3.1-8B

In [ ]:
!python scripts/04_train_qlora.py --model llama8b
!python scripts/03_run_eval.py --model llama8b --strategy qlora --adapter adapters/llama8b_r16_s42
!python scripts/03_run_eval.py --model llama8b --strategy qlora --adapter adapters/llama8b_r16_s42 --no-docs

### 5b. Mistral-7B

In [ ]:
!python scripts/04_train_qlora.py --model mistral7b
!python scripts/03_run_eval.py --model mistral7b --strategy qlora --adapter adapters/mistral7b_r16_s42
!python scripts/03_run_eval.py --model mistral7b --strategy qlora --adapter adapters/mistral7b_r16_s42 --no-docs

### 5c. Qwen2.5-7B

In [ ]:
!python scripts/04_train_qlora.py --model qwen7b
!python scripts/03_run_eval.py --model qwen7b --strategy qlora --adapter adapters/qwen7b_r16_s42
!python scripts/03_run_eval.py --model qwen7b --strategy qlora --adapter adapters/qwen7b_r16_s42 --no-docs

## Step 6. Results
Run this at any time. It uses whatever has finished so far: the summary table, the statistical tests (once a model has two or more strategies), the TOPSIS ranking and the charts.

In [ ]:
!python scripts/05_analyze.py
!python scripts/06_plot_results.py
import pandas as pd
pd.read_csv('results/summary.csv')

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=900))

## Step 7. Save everything to GitHub now
Results already save themselves while a model runs. Run this after Step 6 (to save the summary and charts) or whenever you want to be sure everything is saved.

In [ ]:
import sys, os
for p in [os.path.join(os.getcwd(), 'src'), '/kaggle/working/project/src', '/content/project/src']:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)   # lets this notebook import the project code (kis) without a restart
from kis.autosave import autosave
import datetime
autosave(f"Main study results ({datetime.date.today()})") or print("Nothing saved: run Setup A first, or check the message above.")